In [17]:
import polars as pl
import requests


LINKS = 'webapp/static/links.csv'
MOVIES = 'webapp/static/movies.csv'
INTERACTIONS = 'data/interactions.csv'

In [ ]:
HOST = 'http://157.180.37.79'
recommendation_service_url = f"{HOST}:5001"
movie_id_title = {'1':'Funny Bunny'}

def init_items():
    item_ids = list(movie_id_title.keys())
    url = f'{recommendation_service_url}/add_items'
    message = {'item_ids': item_ids}
    response = requests.post(url, json=message) 
    return response


# res = init_items()

In [16]:
user_id = '0011af63-8f4e-4f2d-a476-4b3f3fdd80b0'
key = f'recomendations:{user_id}'
[i.decode('utf-8') for i in redis_sync_client.lrange(key, 0, -1)]

['86c84664-ca28-4928-baf6-9a94dabe9e7d',
 '878d264d-d3be-4a41-8943-5060c04a7dfc',
 '5014b36b-95cb-42b8-a2b5-a8e481153529',
 'ff6354b9-abf8-4f57-8615-76c5c1c4d138',
 'da8174b7-8d76-4036-912a-84c4a86994cb',
 'e88ab673-dbca-4d84-a78f-28878b9f346f',
 '645ce29a-0c61-46f3-8e15-9e13afed5d28',
 '067d4687-be83-4e64-8349-e2de681fe5f9',
 '2e514a3f-76be-47f5-94b1-5e5ed8c81f84',
 '0f84088a-4459-4123-9c76-c02b7e07ee0d']

In [25]:
import asyncio, os
from concurrent.futures import ThreadPoolExecutor
import redis.asyncio as redis_async
import redis
from lightfm import LightFM
import polars as pl, numpy as np
from scipy.sparse import coo_matrix
from typing import List

# current_dir = os.path.dirname(os.path.abspath(__file__))
current_dir = './recommendations/'
interactions_path = os.path.join(current_dir, '..', 'data', 'interactions.csv')

redis_async_client = None
redis_sync_client = None
executor = ThreadPoolExecutor(max_workers=2)

async def get_redis_async():
    global redis_async_client
    if redis_async_client is None:
        redis_async_client = await redis_async.from_url("redis://localhost")
    return redis_async_client
    
def get_redis_sync():
    global redis_sync_client
    if redis_sync_client is None:
        redis_sync_client = redis.Redis('localhost')
    return redis_sync_client

def get_mapping(df, col):
    df = df.select(pl.col(col)).unique()
    df = df.with_row_count('int_map', offset=0)
    return {
        'forward': dict(zip(df[col], df['int_map'])),
        'backward': dict(zip(df['int_map'], df[col])),
    }

def save_mapping(mapping, bucket='users'):
    r = get_redis_sync()
    for direction in ['forward', 'backward']:
        r.json().set(f"mapping:{bucket}:{direction}", "$", mapping[direction])
    print(f'Mapping save to bucket {bucket}')
    return 


def load_interactions():
    global interactions_path
    interact_summary = (pl.read_csv(interactions_path)
                          .with_columns(pl.when(pl.col('action')=='like').then(1).otherwise(-1).alias('score'))
                          .groupby('user_id', 'item_id')
                          .agg(pl.sum('score')))
    
    user_mapping = get_mapping(interact_summary, 'user_id')
    item_mapping = get_mapping(interact_summary, 'item_id')
    
    save_mapping(user_mapping, bucket='users')
    save_mapping(item_mapping, bucket='items')
    
    interact_summary = (interact_summary.with_columns(pl.col("user_id").map_dict(user_mapping['forward']).alias('user_id'),
                                                      pl.col("item_id").map_dict(item_mapping['forward']).alias('item_id'),))
    users = interact_summary['user_id'].to_list()
    items = interact_summary['item_id'].to_list()
    scores = interact_summary['score'].to_list()
    
    N_USERS = max(users) + 1
    N_ITEMS = max(items) + 1
    
    interactions = coo_matrix((scores, (users, items)), shape=(N_USERS, N_ITEMS))
    print('Interaction loaded ...')
    return interactions
    
def train_model(data,
                learning_rate=0.05,
                no_components=30,
                random_state=42,
                epochs=30,
                num_threads=2,
                verbose=True):
    
    model = LightFM(loss='warp',  
                    learning_rate=learning_rate,
                    no_components=no_components,  
                    random_state=random_state)
    
    model.fit(data, epochs=epochs, num_threads=num_threads, verbose=verbose)
    print('Model builded ...')
    return model
    
def get_top_n_items_for_all_users(model:LightFM, interactions:coo_matrix, n_items=10, num_threads=4):
    n_users, n_items_total = interactions.shape

    scores = model.predict(
        user_ids=np.repeat(np.arange(n_users), n_items_total),
        item_ids=np.tile(np.arange(n_items_total), n_users),
        num_threads=num_threads
    ).reshape(n_users, n_items_total)
    
    top_items = np.argsort(-scores, axis=1)[:, :n_items]
    
    recommendations = {}
    for user_id in range(n_users):
        recommendations[user_id] = top_items[user_id].tolist()
        
    print('Recomendations generated ...')
    return recommendations
    
async def load_mapping(bucket='users'):
    r = await get_redis_async()
    result = {}
    
    for direction in ['forward', 'backward']:
        result[direction] = await r.json().get(f"mapping:{bucket}:{direction}")
        
    return {
        'forward' : result['forward'],
        'backward' : result['backward'],
    }
    
async def save_recomendations(recommendations):
    r = await get_redis_async()
    user_mapping_task = load_mapping(bucket='users')
    item_mapping_task = load_mapping(bucket='items')
    
    user_mapping = (await user_mapping_task)['backward']
    item_mapping = (await item_mapping_task)['backward']
    
    for user, rec in recommendations.items():
        user_origin = user_mapping[str(user)]
        rec_origin = [item_mapping[str(i)] for i in rec]
        key = f'recomendations:{user_origin}'
        await r.delete(key)  
        await r.rpush(key, *rec_origin)
    print('Recomendations saved ...')
    return 
    
async def run(top_k=10):
    print('Start train process ...')
    executor = ThreadPoolExecutor(max_workers=2)
    loop = asyncio.get_event_loop()
    interactions = await loop.run_in_executor(executor, load_interactions)
    model = await loop.run_in_executor(executor, lambda: train_model(interactions) ) 
    recommendations = await loop.run_in_executor(executor, lambda: get_top_n_items_for_all_users(model, interactions, n_items=top_k))
    await save_recomendations(recommendations) 
    return 

await run()

Start train process ...
Mapping save to bucket users
Mapping save to bucket items
Interaction loaded ...


Epoch: 100%|██████████| 30/30 [00:00<00:00, 67.29it/s]


Model builded ...
Recomendations generated ...
Recomendations saved ...


In [31]:
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix

class ALSmodel():
    @staticmethod
    def get_top_n_items_for_all_users(model, user_items_matrix, n_items=10, 
                                      filter_already_liked=True):
    
        # Конвертируем в CSR если это COO матрица
        if not isinstance(user_items_matrix, csr_matrix):
            user_items_matrix = user_items_matrix.tocsr()
        
        n_users, n_items_total = user_items_matrix.shape
        
        recommendations = {}
        
        # ALS модель из implicit не имеет метода predict для всех пар сразу
        # Поэтому используем recommend для каждого пользователя
        for user_id in range(n_users):
            # Получаем топ-N рекомендаций для пользователя
            recommended_items = model.recommend(
                user_id,
                user_items_matrix[user_id],  # теперь это работает для CSR
                N=n_items,
                filter_already_liked_items=filter_already_liked,
                recalculate_user=False,
            )
            
            recommendations[user_id] = recommended_items[0].tolist()
            
            # Прогресс (опционально)
            if (user_id + 1) % 1000 == 0:
                print(f'Processed {user_id + 1}/{n_users} users...')
        
        print(f'Recommendations generated for {n_users} users')
        return recommendations
    @staticmethod    
    def train_model(data,
                    factors=50,
                    regularization=0.1,
                    iterations=20,
                    random_state=42,
                    use_gpu=False):
        
        model = AlternatingLeastSquares(factors=factors,     
                                        regularization=regularization,   
                                        iterations=iterations,        
                                        random_state=random_state,     
                                        use_gpu=use_gpu)
        
        model.fit(interactions)
        print('ALS Model builded ...')
        return model
        
    @staticmethod    
    async def run(top_k=10):
        print('Start train process ...')
        executor = ThreadPoolExecutor(max_workers=2)
        loop = asyncio.get_event_loop()
        interactions = await loop.run_in_executor(executor, load_interactions)
        model = await loop.run_in_executor(executor, lambda: ALSmodel.train_model(interactions) ) 
        recommendations = await loop.run_in_executor(executor, lambda: ALSmodel.get_top_n_items_for_all_users(model, interactions, n_items=top_k))
        await save_recomendations(recommendations) 
        return 
    
await ALSmodel.run(top_k=20)

Start train process ...
Mapping save to bucket users
Mapping save to bucket items
Interaction loaded ...


/usr/local/lib/python3.10/dist-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.001558542251586914 seconds
  warnings.warn(


  0%|          | 0/20 [00:00<?, ?it/s]

ALS Model builded ...
Processed 1000/8102 users...
Processed 2000/8102 users...
Processed 3000/8102 users...
Processed 4000/8102 users...
Processed 5000/8102 users...
Processed 6000/8102 users...
Processed 7000/8102 users...
Processed 8000/8102 users...
Recommendations generated for 8102 users
Recomendations saved ...


In [1]:
from lightfm import LightFM
import numpy as np
import polars as pl
from scipy.sparse import coo_matrix
import redis
from typing import List

INTERACTIONS = 'data/interactions.csv'

In [5]:
def get_mapping(df, col):
    df = df.select(pl.col(col)).unique()
    df = df.with_row_count('int_map', offset=0)
    return {
        'forward': dict(zip(df[col], df['int_map'])),
        'backward': dict(zip(df['int_map'], df[col])),
    }

def save_mapping(mapping, bucket='users'):
    r = redis.Redis()
    for direction in ['forward', 'backward']:
        r.json().set(f"mapping:{bucket}:{direction}", "$", mapping[direction])
    print(f'Mapping save to bucket {bucket}')
    return 

def load_mapping(bucket='users'):
    r = redis.Redis()
    result = {}
    
    for direction in ['forward', 'backward']:
        result[direction] = r.json().get(f"mapping:{bucket}:{direction}")
        
    return {
        'forward' : result['forward'],
        'backward' : result['backward'],
    }

def load_interactions():
    global INTERACTIONS
    interact_summary = (pl.read_csv(INTERACTIONS)
                          .with_columns(pl.when(pl.col('action')=='like').then(1).otherwise(-1).alias('score'))
                          .groupby('user_id', 'item_id')
                          .agg(pl.sum('score')))
    
    user_mapping = get_mapping(interact_summary, 'user_id')
    item_mapping = get_mapping(interact_summary, 'item_id')
    
    save_mapping(user_mapping, bucket='users')
    save_mapping(item_mapping, bucket='items')
    
    interact_summary = (interact_summary.with_columns(pl.col("user_id").map_dict(user_mapping['forward']).alias('user_id'),
                                                      pl.col("item_id").map_dict(item_mapping['forward']).alias('item_id'),))
    users = interact_summary['user_id'].to_list()
    items = interact_summary['item_id'].to_list()
    scores = interact_summary['score'].to_list()
    
    N_USERS = max(users) + 1
    N_ITEMS = max(items) + 1
    
    interactions = coo_matrix((scores, (users, items)), shape=(N_USERS, N_ITEMS))
    print('Interaction loaded ...')
    return interactions

def train_model(data,
                learning_rate=0.05,
                no_components=30,
                random_state=42,
                epochs=30,
                num_threads=2,
                verbose=True):
    
    model = LightFM(loss='warp',  
                    learning_rate=learning_rate,
                    no_components=no_components,  
                    random_state=random_state)
    
    model.fit(data, epochs=epochs, num_threads=num_threads, verbose=verbose)
    print('Model builded ...')
    return model

def get_top_n_items_for_all_users(model, interactions, n_items=10, num_threads=4):
    n_users, n_items_total = interactions.shape

    scores = model.predict(
        user_ids=np.repeat(np.arange(n_users), n_items_total),
        item_ids=np.tile(np.arange(n_items_total), n_users),
        num_threads=num_threads
    ).reshape(n_users, n_items_total)
    
    top_items = np.argsort(-scores, axis=1)[:, :n_items]
    
    recommendations = {}
    for user_id in range(n_users):
        recommendations[user_id] = top_items[user_id].tolist()
        
    print('Recomendations generated ...')
    return recommendations
    
def save_recomendations(recommendations):
    r = redis.Redis()
    user_mapping = load_mapping(bucket='users')['backward']
    item_mapping = load_mapping(bucket='items')['backward']
    
    for user, rec in recommendations.items():
        user_origin = user_mapping[str(user)]
        rec_origin = [item_mapping[str(i)] for i in rec]
        key = f'recomendations:{user_origin}'
        r.delete(key)  
        r.rpush(key, *rec_origin)
    print('Recomendations saved ...')
    return 

def get_recomendation(user_id)->List:
    r = redis.Redis()
    key = f'recomendations:{user_id}'
    rec = r.get(key)
    return rec

def run(top_k=10):
    print('Start train process ...')
    # load coo matrix
    interactions = load_interactions() 
    # LightFM model
    model = train_model(interactions) 
    # build recomendations for each user
    recommendations = get_top_n_items_for_all_users(model, interactions, n_items=top_k) 
    # save to redis
    save_recomendations(recommendations) 
    return 

run()

Start train process ...
Mapping save to bucket users
Mapping save to bucket items
Interaction loaded ...
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Epoch 15
Epoch 16
Epoch 17
Epoch 18
Epoch 19
Epoch 20
Epoch 21
Epoch 22
Epoch 23
Epoch 24
Epoch 25
Epoch 26
Epoch 27
Epoch 28
Epoch 29
Model builded ...
Recomendation generated ...
Recomendation saved ...
